# GC-TGCUP on Kaggle GPU — beat TG-CUP

**Before running:** Notebook Settings → Accelerator = **GPU T4 x2** (or P100), Internet = **ON** (GraphCodeBERT downloads ~500 MB).

Flow: clone the small GitHub repo (code only) → symlink the uploaded `cup2_dataset` → prepare data → train → evaluate. The 20 GB dataset stays in `/kaggle/input` (read-only), so the working disk does NOT fill up.

In [ ]:
# 1) Clone the code repo (no data inside it)
!rm -rf /kaggle/working/gctgcup
!git clone https://github.com/MalihaIlyas80/gctgcup.git /kaggle/working/gctgcup
%cd /kaggle/working/gctgcup

In [ ]:
# 2) Install deps (torch already present on Kaggle)
!pip install -q transformers sacrebleu nltk scikit-learn pyyaml tqdm javalang

In [ ]:
# 3) Locate the uploaded cup2_dataset and symlink it (no copy = no extra storage)
import os, glob

# Find a folder named cup2_dataset anywhere under /kaggle/input
candidates = [p for p in glob.glob('/kaggle/input/**/cup2_dataset', recursive=True)]
assert candidates, 'cup2_dataset not found under /kaggle/input — add your dataset to the notebook.'
CUP_DATASET_PATH = candidates[0]
print('Found:', CUP_DATASET_PATH)

link = '/kaggle/working/gctgcup/cup2_dataset'
if os.path.lexists(link):
    os.remove(link)
os.symlink(CUP_DATASET_PATH, link)
print('Linked ->', os.listdir(link))

# Keep HuggingFace cache off the working disk
os.environ['HF_HOME'] = '/kaggle/temp/hf'
os.makedirs('/kaggle/temp/hf', exist_ok=True)

In [ ]:
# 4) Prepare data: clean + closed vocab + stratified split
#    Raise --max-samples for the final run (e.g. 40000) if time allows.
!python scripts/prepare_data.py --max-samples 20000 --vocab-max-size 30000

In [ ]:
# 5) Train (GraphCodeBERT fine-tuned). Resumes from checkpoints/best.pt if present.
!python scripts/train.py --config configs/kaggle_gpu.yaml

In [ ]:
# 6) Evaluate vs TG-CUP and show the comparison
!python scripts/evaluate.py --config configs/kaggle_gpu.yaml --checkpoint checkpoints/best.pt

In [ ]:
# 7) Save small outputs you can download from the right sidebar
import shutil, os, json
os.makedirs('/kaggle/working/output', exist_ok=True)
for f in ['best.pt', 'evaluation_report.json', 'training_history.json']:
    src = f'checkpoints/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/kaggle/working/output/{f}')
rep = 'checkpoints/evaluation_report.json'
if os.path.exists(rep):
    print(json.dumps(json.load(open(rep)), indent=2))
print('\nDownload /kaggle/working/output (best.pt + reports).')